## Generative models for modality integration

In [ ]:
import os
import sys
import gc

import tifffile
import numpy as np
import pandas as pd
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from ml_collections import ConfigDict

from skimage import io
from skimage.transform import rescale
from skimage.exposure import rescale_intensity
from skimage.color import rgb2gray
# from denoising_diffusion_pytorch import Unet, GaussianDiffusion, Trainer

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

### Load H&E images

In [ ]:
%ls ../data/he_imgs/res

In [ ]:
data_path = '../data/he_imgs/res/'
img_stacked = tifffile.imread(os.path.join(data_path, 'valis_stacked.ome.tif'))
img_stacked.shape

In [ ]:
# Trim artifacts

_, nimg, ylen, xlen = img_stacked.shape
side_len = 256*10
ymin, ymax = ylen//2-side_len//2, ylen//2+side_len//2
xmin, xmax = xlen//2-side_len//2, xlen//2+side_len//2

ncol = 4
nrow = nimg // ncol
if nimg % ncol != 0:
    nrow += 1

fig, axes = plt.subplots(nrow, ncol, figsize=(3*ncol, 3.2*nrow), dpi=200)
idx = 0
for r in range(nrow):
    for c in range(ncol):
        if idx < nimg:
            axes[r, c].imshow(img_stacked[:, idx, ymin:ymax, xmin:xmax].transpose(1,2,0))
        else:
            axes[r, c].axis('off')
        idx += 1
        
plt.tight_layout()
plt.show()
del idx, r, c

In [ ]:
# Generate training images
img_size = 256
scale=img_size/side_len

imgs = [rescale(img_stacked[:, i, ymin:ymax, xmin:xmax].transpose(1,2,0),
               scale=scale, preserve_range=True, channel_axis=-1).astype(np.uint8)
        for i in range(nimg)]

In [ ]:
fig, axes = plt.subplots(nrow, ncol, figsize=(3*ncol, 3.2*nrow), dpi=200)
idx = 0
for r in range(nrow):
    for c in range(ncol):
        if idx < nimg:
            axes[r, c].imshow(imgs[idx])
        else:
            axes[r, c].axis('off')
        idx += 1
        
plt.tight_layout()
plt.suptitle('Low-res H&E slices', y=1.01)
plt.show()
del idx, r, c

In [ ]:
train_path = '../data/he_imgs/ddpm_train'
if not os.path.exists(train_path):
    os.makedirs(train_path, exist_ok=True)

for i, img in enumerate(imgs):
    id = '0'+str(i) if i < 10 else str(i)
    
    # Save as tif
    # filename = 'he_{}.tiff'.format(id)
    # tifffile.imwrite(os.path.join(train_path, filename), 
    #                  rescale_intensity(imgs_ds[i], out_range=(0, 1)))

    # Save as png
    filename = 'he_{}.png'.format(id)
    io.imsave(os.path.join(train_path, filename), img)

### Train standard DDPM model

In [ ]:
base_dim = 64
layer_mults = (1,2,3,4)
img_size = 256
timesteps = 500
sampling_timesteps = 250

batch_size = 2
lr = 1e-5
nepoch = 5000
ema_decay = 0.999

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model = Unet(dim=base_dim, dim_mults=layer_mults).to(device)

diffusion = GaussianDiffusion(model,
                              image_size=img_size,
                              timesteps=timesteps,
                              sampling_timesteps=sampling_timesteps).to(device)

In [ ]:
trainer = Trainer(diffusion,
                  folder='../data/he_imgs/ddpm_train/',
                  train_batch_size=batch_size,
                  train_lr=lr,
                  train_num_steps=nepoch,
                  gradient_accumulate_every=16,
                  ema_decay=ema_decay,
                  amp=True,
                  calculate_fid=False)


In [ ]:
trainer.train()

In [ ]:
trainer.save(milestone=1000)

#### Interpolation btw adjacent slices:

- Input dimension converted to: `(B, C, Y, X)`
- Pixel intensities normalized to [0, 1] per channel

In [ ]:
# Load model
trainer.load(10)

In [ ]:
x0 = io.imread('../data/he_imgs/ddpm_train/he_04.png')
x1 = io.imread('../data/he_imgs/ddpm_train/he_05.png')

In [ ]:
def interp_z_slice(
    ddpm_model,
    x0_rgb,
    x1_rgb,
    device=None,
    t=None,
    alpha=0.5
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ddpm_model = ddpm_model.to(device)
    
    # Norm by channel, convert to (B, C, Y, X) tensors
    x0 = np.array([rescale_intensity(chan, out_range=(0, 1))
                   for chan in x0_rgb])
    x0 = torch.tensor(x0.transpose(2, 0, 1)).unsqueeze(0).float().to(device)
    x1 = np.array([rescale_intensity(chan, out_range=(0, 1))
                   for chan in x1_rgb])
    x1 = torch.tensor(x1.transpose(2, 0, 1)).unsqueeze(0).float().to(device)

    # Interpolate, convert back to (Y, X, C)
    x0_x1_interp = ddpm_model.interpolate(x0, x1, lam=alpha)
    x0_x1_interp = x0_x1_interp.detach().cpu().squeeze().numpy().transpose(1,2,0)

    # DEBUG: Weird bug flipping the img somehow...
    x0_x1_interp = np.flip(x0_x1_interp, axis=1)

    return np.round(x0_x1_interp*255).astype(np.uint8)
    

In [ ]:
x0 = io.imread('../data/he_imgs/ddpm_train/he_04.png')
x1 = io.imread('../data/he_imgs/ddpm_train/he_05.png')
x0_x1_interp = interp_z_slice(diffusion, x0, x1, device=device)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3))
ax1.imshow(rgb2gray(x0))
ax2.imshow(rgb2gray(x1))
ax3.imshow(rgb2gray(x0_x1_interp))
plt.tight_layout()
plt.show()

In [ ]:
# Interpolating w/ different alphas
interps = []
alphas = np.linspace(0, 1, 6)[1:-1]
for alpha in alphas:
    interp = interp_z_slice(diffusion, x0, x1, device=device)
    interp = rgb2gray(interp)
    interps.append(interp)

In [ ]:
fig, axes = plt.subplots(1, len(alphas), figsize=(2*len(alphas), 2.2))
for (ax, alpha, interp) in zip(axes, alphas, interps):
    ax.imshow(interp)
    ax.set_title(r'Interp ($\alpha={}$)'.format(np.round(alpha, 1)))
plt.tight_layout()
plt.show()

### Load CyIF images

- Create 3-channel zonation-informed input: `GS` + `CYP3A4` + `ASS`
- Create low-dim noisy estimation on zonation dynamics $p(z)$ w/ determinsitic heat diffusion

In [ ]:
from skimage.exposure import equalize_adapthist, rescale_intensity
from skimage.transform import rescale
from skimage.filters import sobel, threshold_otsu, threshold_minimum
from skimage.filters import gaussian as gaussian_blur

sys.path.append('..')
import IO, zonation

from importlib import reload
reload(IO)
reload(zonation)

In [ ]:
data_path = '../data/cycif/Cy-IF_downsample_adj/'
zone_labels = ['GS 647', 'CYP3A4', 'ASS1 PE']

In [ ]:
cyif_annot_imgs, filenames = IO.load_annot_tiffs(data_path, ext='ome.tif')
imgs = [np.array([img[label] for label in zone_labels])
        for img in cyif_annot_imgs]

# Trim middle FOV
radius = 2048

imgs_trimmed = []
for img in imgs:
    ny, nx = img.shape[-2:]
    ylow = ny//2 - radius
    yhigh = ny//2 + radius
    imgs_trimmed.append(
        np.array([
            equalize_adapthist(chnl) 
            for chnl in img[:, ylow:yhigh, ylow:yhigh]
        ])
    )

del img, ny, nx, ylow, yhigh

In [ ]:
def disp_cyif_chans(imgs, title=None, ncols=4):
    """Display 3-channel zonation image"""
    depth = len(imgs)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(imgs[idx].transpose(1,2,0))
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
disp_cyif_chans(imgs_trimmed, title='CyIFs (3-chan zonation markers)')

In [ ]:
# Rescale to high-res (512 x 512) images
out_path = '../data/cycif/zonation_hires/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

in_dim = 4096
out_dim = 512
scale = out_dim / in_dim


for filename, img in zip(filenames, imgs_trimmed):
    filename = filename.split('.')[0] + '.tif'
    img_ds = rescale(img, scale=scale, preserve_range=True, channel_axis=0)
    tifffile.imwrite(os.path.join(out_path, filename), img_ds)

del filename, img, img_ds, out_path

# Rescale to low-res (128 x 128) images
out_path = '../data/cycif/zonation_lowres/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

for filename, img in zip(filenames, imgs):
    img_lowres = rescale(img, scale=0.25, preserve_range=True, channel_axis=0)
    tifffile.imwrite(os.path.join(out_path, filename), img_lowres)

del filename, img, img_lowres, out_path

**Heat diffusion on low-dim images**

In [ ]:
# data_path = '../data/cycif/zonation_hires/'
# filenames = [f for f in sorted(os.listdir(data_path))
#              if f[-3:] == 'tif']

# imgs = [tifffile.imread(os.path.join(data_path, f))
#         for f in filenames]


# data_path = '../data/cycif/zonation_lowres/'
# filenames = [f for f in sorted(os.listdir(data_path))
#              if f[-3:] == 'tif']

# imgs = [tifffile.imread(os.path.join(data_path, f))
#         for f in filenames]

Generate 64 x 64 low-res heat diffusion: has to be run on hi-res image (o.w. GS signal loss...)


In [ ]:
def create_vein_prior(img, sigma=1.5):
    assert img.ndim == 3 and img.shape[0] == 3, "3-dim zonation marker images"
    cv_chan = gaussian_blur(img[0], sigma=sigma)
    cv_threshold = threshold_otsu(cv_chan)
    cv_mask = (cv_chan > cv_threshold).astype(np.uint8)

    pv_chan = gaussian_blur(img[-1], sigma=sigma)
    pv_threshold = threshold_otsu(pv_chan)
    pv_mask = (pv_chan > pv_threshold).astype(np.uint8)

    vein_prior = np.zeros_like(cv_chan, dtype=np.int8)
    vein_prior[np.logical_and(pv_mask == 1, cv_mask == 0)] = -1
    vein_prior[np.logical_and(pv_mask == 0, cv_mask == 1)] = 1

    return vein_prior
    

In [ ]:
vein_path = '../data/cycif/zonation_mask/'
# if not os.path.exists(vein_path):
#     os.makedirs(vein_path, exist_ok=True)

# vein_priors = []
# for filename, img in zip(filenames, imgs):
#     vein_prior = create_vein_prior(img)
#     vein_priors.append(vein_prior)

#     filename = 'CyIF_vein_prior_{}.tif'.format(filename.split('.')[0][-2:])
#     tifffile.imwrite(os.path.join(vein_path, filename), vein_prior)

# del filename, img, vein_prior

# Load from file
# ...

In [ ]:
temps = []
for vein_prior in vein_priors:
    model = zonation.HeatDiffusion(vein_prior=vein_prior, ndim=2)
    U_i, interior_nodes = model.get_interior_U()
    U = model.inference_zone_dynamics()
    temps.append(U)


[2024-01-11 04:10:54] Initializing boundary temperature `U_b`...
[2024-01-11 04:10:54] Inferring `interior node` temperature `U_i`...
[2024-01-11 04:10:59] Projecting temperature {U_b, U_i} back to image space...
[2024-01-11 04:10:59] Creating 2D graph w/ dimension (512, 512)...
[2024-01-11 04:11:01] Initializing boundary temperature `U_b`...
[2024-01-11 04:11:01] Inferring `interior node` temperature `U_i`...
[2024-01-11 04:11:08] Projecting temperature {U_b, U_i} back to image space...
[2024-01-11 04:11:08] Creating 2D graph w/ dimension (512, 512)...
[2024-01-11 04:11:09] Initializing boundary temperature `U_b`...
[2024-01-11 04:11:10] Inferring `interior node` temperature `U_i`...
[2024-01-11 04:11:16] Projecting temperature {U_b, U_i} back to image space...
[2024-01-11 04:11:16] Creating 2D graph w/ dimension (512, 512)...
[2024-01-11 04:11:17] Initializing boundary temperature `U_b`...
[2024-01-11 04:11:17] Inferring `interior node` temperature `U_i`...
[2024-01-11 04:11:23] Proj

In [ ]:
def disp_zone_temps(imgs, title=None, ncols=4):
    """Display 3-channel zonation image"""
    depth = len(imgs)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(imgs[idx], cmap='coolwarm')
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
# Save downscaled (64 x 64) zonation priors as p(z)

vein_path = '../data/cycif/zonation_mask/'
if not os.path.exists(vein_path):
    os.makedirs(vein_path, exist_ok=True)

# in_dim = 512
# out_dim = 64
# scale= out_dim / in_dim
# pz_means = []

# for filename, temp in zip(filenames, temps):
#     pz_mean = rescale(temp, scale=scale, preserve_range=True)
#     pz_means.append(pz_mean)

#     filename = 'CyIF_dynamics_{}.tif'.format(filename.split('.')[0][-2:])
#     tifffile.imwrite(os.path.join(vein_path, filename), pz_mean)


# del filename, temp, pz_mean

# Load from file
# ...

In [ ]:
disp_zone_temps(pz_means, title=r'CyIF zonation priors $p(z)$')

### Vanilla $\beta$-VAE for Cy-IF reconstruction

In [ ]:
import torch.optim as optim
from torch.distributions import Normal, kl_divergence
from torch.utils.data import Dataset, DataLoader


Set prior std $\sigma$ as fixed quantity:

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, c_in, c_out, p=0.1):
        super(ResidualBlock, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(4, c_out),
            nn.SiLU(),
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(c_out, c_out, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(4, c_out)
        )

        self.drop_layer = nn.Dropout2d(p)
        self.skip_layer = nn.Conv2d(c_in, c_out, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        x_resid = self.layer1(x)
        x_resid = self.layer2(x_resid)
        x_skip = self.skip_layer(x)
        x = x_resid + x_skip
        out = self.drop_layer(x)
        return out
    

class Encoder(nn.Module):
    def __init__(
        self,
        configs,
    ):
        super(Encoder, self).__init__()
        c_in = configs.c_in
        c_base = configs.c_base
        p = configs.drop_rate
        c_hiddens = [c_base * n 
                     for n in configs.layer_mults]
        
        self.layers = nn.ModuleList([
            ResidualBlock(c_in, c_hiddens[0], p=p)
        ])
        for i in range(len(c_hiddens)-1):
            self.layers.append(
                ResidualBlock(c_hiddens[i], c_hiddens[i+1], p=p)
            )

        self.downscale_layer = nn.MaxPool2d(2)
        
    def forward(self, x):
        x_outs = []
        for layer in self.layers:
            x_out = layer(x)
            x_outs.append(x_out)
            x = self.downscale_layer(x_out)

        z = x.reshape(x.shape[0], -1)
        return x_outs, z
        

class Decoder(nn.Module):
    def __init__(
        self,
        configs,
    ):
        super(Decoder, self).__init__()
        c_base = configs.c_base
        p = configs.drop_rate
        c_hiddens = [c_base * n
                     for n in configs.layer_mults[::-1]]
        c_hiddens.insert(0, c_hiddens[0])

        self.layers = nn.ModuleList([
            ResidualBlock(c_hiddens[i]+c_hiddens[i+1], c_hiddens[i+1], p=p)
            for i in range(len(c_hiddens)-1)
        ])
        
    def forward(self, x_outs, x):
        for layer, x_out in zip(self.layers, x_outs[::-1]):
            x = F.interpolate(x, scale_factor=(2, 2), mode='bilinear') 
            x_concat = torch.cat((x, x_out), axis=1)  # shape: [B, C1+C2, Y, X]
            x = layer(x_concat)
        
        return x


class BetaVAE(nn.Module):
    def __init__(
        self,
        configs,
        prior_std: float = 0.01  
    ):
        super(BetaVAE, self).__init__()
        c_out = configs.c_out
        nlayers = len(configs.layer_mults)
        device = configs.device
        
        self.device = configs.device
        self.batch_size = configs.batch_size
        self.beta = configs.beta  # weights for B-VAE
        self.pz_std = torch.tensor(prior_std).to(device)

        self.c_bn = configs.c_base * configs.layer_mults[-1]

        ny_in, nx_in = configs.ydim, configs.xdim
        self.ny_bn = ny_in // (2**nlayers)
        self.nx_bn = nx_in // (2**nlayers)
        
        # Encoder
        self.encoder = Encoder(configs)
        self.enc_z_mu = nn.Sequential(
            nn.Linear(self.c_bn*self.ny_bn*self.nx_bn, configs.latent_dim),
            nn.Tanh()
        )

        self.enc_z_logvar = nn.Linear(self.c_bn*self.ny_bn*self.nx_bn, configs.latent_dim)

        # Decoder
        self.dec_z_to_hidden = nn.Linear(configs.latent_dim, self.c_bn*self.ny_bn*self.nx_bn, configs.latent_dim)
        self.decoder = Decoder(configs)
        self.out_layer = nn.Conv2d(configs.c_base*configs.layer_mults[0], c_out, kernel_size=1, stride=1)

    def inference(self, x):
        x_encs, z = self.encoder(x)
        qz_mu = self.enc_z_mu(z.flatten())
        qz_logvar = self.enc_z_logvar(z.flatten())
        qz = Normal(qz_mu, torch.exp(0.5*qz_logvar)).rsample()

        inference_terms = ConfigDict()
        inference_terms.qz = qz
        inference_terms.qz_mu = qz_mu
        inference_terms.qz_logvar = qz_logvar
        inference_terms.x_encs = x_encs

        return inference_terms

    def generative(self, x_encs, qz):
        hidden = self.dec_z_to_hidden(qz)
        hidden = hidden.view(self.batch_size, self.c_bn, self.ny_bn, self.nx_bn)
        px_z = self.decoder( x_encs, hidden)
        x_hat = self.out_layer(px_z)
        return torch.sigmoid(x_hat)

    def get_loss(self, x, x_hat, pz_mu, inference_terms):
        # Reconstruction loss
        mse = nn.MSELoss(reduction='none')
        loss_NLL = mse(x, x_hat).sum((1,2,3)).mean()

        # KL divergence
        pz_mu = pz_mu.squeeze().flatten()
        pz_std = torch.ones_like(pz_mu) * self.pz_std

        qz_mu = inference_terms.qz_mu
        qz_logvar = inference_terms.qz_logvar


        loss_KL = kl_divergence(
            Normal(qz_mu, torch.exp(0.5*qz_logvar)),
            Normal(pz_mu, pz_std)
        ).sum(-1).mean()

        loss_configs = ConfigDict()
        loss_configs.tot = (1-self.beta)*loss_NLL + self.beta*loss_KL
        loss_configs.nll = loss_NLL
        loss_configs.kl = loss_KL

        return loss_configs 


In [ ]:
class CyIFDataset(Dataset):
    def __init__(
        self,
        data_path,
        prior_path
    ):
        # TODO: write I/O loading from gcloud
        self.img_names = [
            os.path.join(data_path, f) 
            for f in sorted(os.listdir(data_path))
            if f[-3:] == 'tif' or f[-4:] == 'tiff'
        ]

        self.prior_names = [
            os.path.join(prior_path, f)
            for f in sorted(os.listdir(prior_path))
            if (f[-3:] == 'tif' or f[-4:] == 'tiff') and 'dynamic' in f  # TODO: refactor this!
        ]

        assert len(self.img_names) == len(self.prior_names), "Unequal data & prior sizes:{0}, {1}".format(
            len(self.img_names), len(self.prior_names)
        )

    def __len__(self):
        return len(self.img_names)
    
    def __getitem__(self, index):
        img = tifffile.imread(self.img_names[index])
        pz_mu = tifffile.imread(self.prior_names[index])
        return torch.tensor(img), torch.tensor(pz_mu)



Model training functions:

In [ ]:
def run_one_epoch(model, dataloader, optimizer, device):
    model = model.to(device)
    model.train()

    loss_sum = 0.0
    nll_sum = 0.0
    kl_sum = 0.0    

    cnt = 0
    for x, pz_mu in dataloader:
        
        cnt += 1
        x = x.float().to(device)
        pz_mu = pz_mu.float().to(device)
        
        inference_terms = model.inference(x)
        x_pred = model.generative(inference_terms.x_encs, 
                                  inference_terms.qz)
                                  
        if any(torch.isnan(p).any() for p in model.parameters()):
            print('NaNs detected in model parameters, Skipping current epoch...')
            continue    

        loss_configs = model.get_loss(x, x_pred, pz_mu, inference_terms)
        loss, loss_nll, loss_kl = loss_configs.tot, loss_configs.nll, loss_configs.kl

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()

        loss_sum += loss
        nll_sum += loss_nll
        kl_sum = loss_kl

    avg_loss = loss_sum / cnt
    avg_nll = nll_sum / cnt
    avg_kl = kl_sum / cnt

    return avg_loss, avg_nll, avg_kl


def train(train_configs, model_configs):
    torch.manual_seed(0)
    np.random.seed(0)

    losses = []
    losses_nll = []
    losses_kl = []

    dataset = CyIFDataset(data_path=train_configs.data_path, prior_path=train_configs.prior_path)
    dataloader = DataLoader(dataset, batch_size=model_configs.batch_size, shuffle=True, drop_last=True)

    device = model_configs.device
    model = BetaVAE(model_configs, prior_std=model_configs.pz_std)
    optimizer = optim.Adam(model.parameters(), lr=train_configs.lr)
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)

    n_epochs = train_configs.n_epochs
    for epoch in range(n_epochs):
        avg_loss, avg_nll, avg_kl = run_one_epoch(model, dataloader, optimizer, device=device)
        losses.append(avg_loss)
        losses_nll.append(avg_nll)
        losses_kl.append(avg_kl)

        scheduler.step()

        if (epoch + 1) % 10 == 0:
            print("Epoch[{}/{}], total_loss: {:.4f}, reconst: {:.4f}, kl: {:.4f}".format(
                epoch + 1, n_epochs, avg_loss, avg_nll, avg_kl)
            )

        torch.cuda.empty_cache()

    losses_dict = {
        'total': losses,
        'NLL': losses_nll,
        'KL': losses_kl
    }

    return model, losses_dict


#### Training

In [ ]:
# Model configs
model_configs = ConfigDict()
model_configs.c_in = 3
model_configs.c_out = 3
model_configs.c_base = 4
model_configs.layer_mults = [1, 2]
model_configs.drop_rate = 0.1

model_configs.device = torch.device('cpu')
model_configs.batch_size = 1
model_configs.beta = 0.1

model_configs.ydim = 128
model_configs.xdim = 128
model_configs.latent_dim = 4096

model_configs.pz_std = 0.01

# model_configs.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_configs.device = torch.device('cpu')


In [ ]:
train_configs = ConfigDict()

# train_configs.data_path = '../data/cycif/zonation_hires/'
train_configs.data_path = '../data/cycif/zonation_lowres/'
train_configs.prior_path = '../data/cycif/zonation_mask/'
train_configs.lr = 1e-5
train_configs.n_epochs = 200


In [ ]:
model, loss_dict = train(train_configs, model_configs)

In [ ]:
loss_dict.keys()

In [ ]:
loss_tot = torch.tensor(loss_dict['total']).detach().cpu().numpy()
loss_nll = torch.tensor(loss_dict['NLL']).detach().cpu().numpy()
loss_kl = torch.tensor(loss_dict['KL']).detach().cpu().numpy()

In [ ]:
np.arange(n_epochs)[::step]

In [ ]:
n_epochs = train_configs.n_epochs
beta = model_configs.beta

step = 10

plt.figure(figsize=(6, 4))
plt.plot(np.arange(n_epochs)[::step], loss_tot[::step], '.--',  label = 'Total loss')
plt.plot(np.arange(n_epochs)[::step], loss_nll[::step], '.--', color='orange', label = r'$\Vert x - \hat{x} \Vert^2$')
plt.plot(np.arange(n_epochs)[::step], loss_kl[::step], '.--', color='green', label = r'$D_{\text{KL}}(q(z \mid x) || p(z))$')

plt.title('Training logs')
plt.legend()
plt.show()


In [ ]:
torch.save(model.state_dict(), 'results/bvae_2d_lowres.pt')

#### Predictions & Assessment

In [ ]:
def predict(model, device):
    model.eval()

    dataset = CyIFDataset(train_configs.data_path, train_configs.prior_path)
    dataloader = DataLoader(dataset, batch_size=1)

    x_list = []
    x_pred_list = []
    pz_list = []
    qz_list = []

    with torch.no_grad():
        for x, pz_mu in dataloader:
            x = x.float().to(device)
            pz_mu = pz_mu.float().to(device)
            x_list.append(x.detach().cpu().numpy().squeeze())
            pz_list.append(pz_mu.detach().cpu().numpy().squeeze())
            
            # Inference
            inference_terms = model.inference(x)
            x_pred = model.generative(inference_terms.x_encs, inference_terms.qz)

            x_pred = x_pred.detach().cpu().numpy().squeeze()
            x_pred_list.append(x_pred)
            
            qz_m = inference_terms.qz_mu.detach().cpu().numpy().squeeze()
            out_dim = np.int8(np.sqrt(len(qz_m)))
            temps_pred = qz_m.reshape(out_dim, -1)
            qz_list.append(temps_pred)

    return x_list, pz_list, x_pred_list, qz_list


In [ ]:
imgs, priors, x_preds, qz_preds = predict(model, device=model_configs.device)

In [ ]:
for i in range(len(x_preds)):
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, dpi=200)
    ax1.imshow(imgs[i].transpose(1,2,0))
    ax1.set_title('Input image')
    ax1.axis('off')
    
    ax2.imshow(x_preds[i].transpose(1,2,0))
    ax2.set_title('Reconstruted image')
    ax2.axis('off')

    ax3.imshow(priors[i], cmap='coolwarm')
    ax3.set_title('Prior dynamics')
    ax3.axis('off')
    
    ax4.imshow(qz_preds[i], cmap='coolwarm')
    ax4.set_title('Predicted dynamics')
    ax4.axis('off')

    plt.tight_layout()
    plt.show()

Interpolation attempts:<br>
Note: Unnecessary short-connections btw input & output

---